# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**What this notebook is.** My model (Lane 4: CTR / engagement opportunity scoring) predicts one thing: **will this visible page still earn far less CTR than its position normally earns in the next 30 days?** (`low_later`, a proxy label). Before any model is trusted, this notebook (1) builds the exact feature vector, (2) writes down what each feature means and when it exists, (3) attacks the features for leakage, and (4) lists what I refused to use.

**The prediction moment:** the end of the *earlier* 30-day window. Anything that needs the *later* window, or that was computed by FlyRank's product from the later window, is the future and cannot be a feature. Observed, decision-support work only; no causal claims.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Scope guard:** only pages with at least 100 impressions in the earlier 30 days and a real position (1-20; `avg_position = 0` means "no data"). **Label:** `low_later` = later-30-day CTR below half of the position band's typical later CTR, defined only for pages with at least 100 later impressions. **Engineered features** compare a page with its band's typical earlier CTR (`ratio`, `gap`, `miss`); the band norms come from **training clients only** inside each fold, so a page's own client never sets its own yardstick.

In [1]:
import os, sys, subprocess, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"
if IN_COLAB:
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

BANDS = [0, 3, 6, 10, 15, 20]; BAND_LABELS = ["1-3", "4-6", "7-10", "11-15", "16-20"]
el = df[(df["impressions_prev_30d"] >= 100) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy().reset_index(drop=True)
el["band"] = pd.cut(el["avg_position"], BANDS, labels=BAND_LABELS)
el["ctr_prev"] = 100 * el["clicks_prev_30d"] / el["impressions_prev_30d"]

# label: needs the LATER window, so it is only ever a target, never an input
el["labelled"] = el["impressions_last_30d"] >= 100
el["ctr_last"] = 100 * el["clicks_last_30d"] / el["impressions_last_30d"]
lt = el[el["labelled"]].groupby("band", observed=True)[["clicks_last_30d", "impressions_last_30d"]].sum()
typ_last = 100 * lt["clicks_last_30d"] / lt["impressions_last_30d"]
el["y"] = np.where(el["labelled"], (el["ctr_last"] < 0.5 * el["band"].map(typ_last).astype(float)).astype(float), np.nan)

# features from the EARLIER window and fixed page properties only
el["log_imp"] = np.log1p(el["impressions_prev_30d"]); el["log_clk"] = np.log1p(el["clicks_prev_30d"])
el["log_sess"] = np.log1p(el["sessions_prev_30d"])
el["has_wc"] = el["word_count"].notna().astype(int); el["wc"] = el["word_count"].fillna(0)      # missing flagged, not hidden
el["has_kw"] = el["search_volume"].notna().astype(int); el["log_sv"] = np.log1p(el["search_volume"].fillna(0))
for v in ["informational", "transactional", "commercial"]:
    el["int_" + v] = (el["main_intent"] == v).astype(int)
el["is_kw_article"] = (el["content_type"] == "keyword article").astype(int)
STATIC = ["log_imp", "log_clk", "ctr_prev", "avg_position", "log_sess", "content_age_days", "has_wc", "wc",
          "has_kw", "log_sv", "int_informational", "int_transactional", "int_commercial", "is_kw_article"]
FEATURES = STATIC + ["ratio", "gap", "miss"]       # last three are added per fold from training-client band norms

def band_typical(train):
    g = train.groupby("band", observed=True)[["clicks_prev_30d", "impressions_prev_30d"]].sum()
    return 100 * g["clicks_prev_30d"] / g["impressions_prev_30d"]

def add_relative(d, typical):
    d = d.copy()
    d["typ"] = d["band"].map(typical).astype(float)
    d["ratio"] = d["ctr_prev"] / d["typ"]; d["gap"] = d["ctr_prev"] - d["typ"]
    d["miss"] = d["impressions_prev_30d"] * d["typ"] / 100 - d["clicks_prev_30d"]
    return d

ev0 = el[el["labelled"]].reset_index(drop=True)
client_fold = {}
for f, (_, te) in enumerate(GroupKFold(n_splits=5).split(ev0, ev0["y"], ev0["client_id"])):
    for c in ev0.iloc[te]["client_id"].unique():
        client_fold[c] = f
el["fold"] = el["client_id"].map(client_fold)

print(f"Eligible pages: {len(el):,} | labelled: {int(el['labelled'].sum()):,} from {el.loc[el['labelled'], 'client_id'].nunique()} clients | base rate {el['y'].mean():.3f}")
print(f"Feature vector: {len(FEATURES)} features -> {FEATURES}")
assert not [c for c in FEATURES if "last_30d" in c or c.startswith("trend_") or c in ("ctr", "days_since_last_update")]

# what the matrix looks like for one fold (training side)
tr0 = add_relative(el[(el["fold"] != 0) & el["labelled"]], band_typical(el[(el["fold"] != 0) & el["labelled"]]))
display(tr0[FEATURES].describe().T[["count", "mean", "min", "max"]].round(2))
print("Missing values left in the feature matrix:", int(tr0[FEATURES].isna().sum().sum()))

Eligible pages: 12,770 | labelled: 11,111 from 29 clients | base rate 0.410
Feature vector: 17 features -> ['log_imp', 'log_clk', 'ctr_prev', 'avg_position', 'log_sess', 'content_age_days', 'has_wc', 'wc', 'has_kw', 'log_sv', 'int_informational', 'int_transactional', 'int_commercial', 'is_kw_article', 'ratio', 'gap', 'miss']


,count,mean,min,max
log_imp,6786.0,7.09,4.62,12.30
log_clk,6786.0,1.48,0.00,7.40
ctr_prev,6786.0,0.36,0.00,6.78
avg_position,6786.0,10.31,0.60,20.00
log_sess,6786.0,2.08,0.00,7.32
content_age_days,6786.0,232.64,90.00,564.00
has_wc,6786.0,0.95,0.00,1.00
wc,6786.0,3186.08,0.00,9157.00
has_kw,6786.0,0.97,0.00,1.00
log_sv,6786.0,1.82,0.00,10.61


Missing values left in the feature matrix: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The test for every row: **does this number exist at the end of the earlier 30-day window?** The table below is checked in code against the data, not just asserted.

In [2]:
notes = pd.DataFrame([
    ["log_imp", "log(1 + earlier-30-day impressions)", "never missing (scope guard needs >=100)", "numeric, log for heavy tail", "yes: earlier window"],
    ["log_clk", "log(1 + earlier-30-day clicks)", "never missing", "numeric, log", "yes: earlier window"],
    ["ctr_prev", "earlier clicks / earlier impressions, in %", "never missing", "numeric", "yes: earlier window"],
    ["avg_position", "average position (1-20 after the scope guard)", "0 = no data, removed by the guard", "numeric", "yes: earlier-period position"],
    ["log_sess", "log(1 + earlier-30-day sessions)", "0 kept as a real zero (a page can get impressions and no sessions)", "numeric, log", "yes: earlier window"],
    ["content_age_days", "days since the page was first seen", "never missing", "numeric", "yes: a fixed property"],
    ["has_wc / wc", "word count and an 'is it present' flag", "missing for about 3 in 10 keyword articles -> filled 0 AND flagged by has_wc", "numeric + flag", "yes: a fixed property"],
    ["has_kw / log_sv", "target keyword search volume (log) and flag", "missing -> filled 0 AND flagged by has_kw", "numeric + flag", "yes: a fixed property"],
    ["int_informational / transactional / commercial", "main search intent", "other or missing -> all three 0", "one-hot", "yes: a fixed property"],
    ["is_kw_article", "content type is 'keyword article'", "never missing", "binary (2 other types are the reference)", "yes: a fixed property"],
    ["ratio / gap / miss", "earlier CTR vs the band's typical earlier CTR (ratio, difference, missing clicks)", "never missing", "numeric; band norms from TRAINING clients only", "yes: earlier window + training clients"],
], columns=["feature", "meaning", "missing values", "type / categorical handling", "exists before the prediction moment?"])
pd.set_option("display.max_colwidth", 80)
display(notes.style.hide(axis="index"))

# the claims in the table, checked against the data
print("word_count missing by content type:", el.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean(), 3)).to_dict())
print("search_volume missing by content type:", el.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean(), 3)).to_dict())
print("main_intent values:", el["main_intent"].value_counts(dropna=False).to_dict())

feature,meaning,missing values,type / categorical handling,exists before the prediction moment?
log_imp,log(1 + earlier-30-day impressions),never missing (scope guard needs >=100),"numeric, log for heavy tail",yes: earlier window
log_clk,log(1 + earlier-30-day clicks),never missing,"numeric, log",yes: earlier window
ctr_prev,"earlier clicks / earlier impressions, in %",never missing,numeric,yes: earlier window
avg_position,average position (1-20 after the scope guard),"0 = no data, removed by the guard",numeric,yes: earlier-period position
log_sess,log(1 + earlier-30-day sessions),0 kept as a real zero (a page can get impressions and no sessions),"numeric, log",yes: earlier window
content_age_days,days since the page was first seen,never missing,numeric,yes: a fixed property
has_wc / wc,word count and an 'is it present' flag,missing for about 3 in 10 keyword articles -> filled 0 AND flagged by has_wc,numeric + flag,yes: a fixed property
has_kw / log_sv,target keyword search volume (log) and flag,missing -> filled 0 AND flagged by has_kw,numeric + flag,yes: a fixed property
int_informational / transactional / commercial,main search intent,other or missing -> all three 0,one-hot,yes: a fixed property
is_kw_article,content type is 'keyword article',never missing,binary (2 other types are the reference),yes: a fixed property


word_count missing by content type: {'comparison article': 0.0, 'feedly article': 0.0, 'keyword article': 0.312}
search_volume missing by content type: {'comparison article': 0.0, 'feedly article': 1.0, 'keyword article': 0.009}
main_intent values: {'informational': 7549, 'transactional': 2843, 'commercial': 2117, nan: 249, 'navigational': 12}


**Reading.** Missingness follows content type (blank word counts are almost all keyword articles; search volume is blank for one type), so a blind zero-fill would quietly teach the model "this is a feed article". That is why every filled field carries an `has_` flag. Every feature comes from the earlier window or is a fixed page property.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Four attacks, each with a number:

1. **Column screen:** the single-column ranking power (AUC, direction-free) of every numeric column, to find anything that looks too good.
2. **Ablation:** drop the suspicious-but-allowed `avg_position`; the score should not collapse.
3. **Adding the forbidden:** add `days_since_last_update` and an outcome-looking trend column; the score tells me whether they were safe.
4. **Controls:** a deliberate leak must score about 1.0 (so the test can detect leaks), and shuffled labels must score about 0.5 (so the pipeline cannot invent signal).

Scores are gradient-boosting AUC on **held-out clients** (5 client-grouped folds).

In [3]:
# ---- Attack 1: column screen on the labelled eligible pages
L = el[el["labelled"]].copy()
yL = L["y"].astype(int).values
num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c not in ("age_tier_order",)]
rows = []
for c in num:
    s = L[c].astype(float)
    if s.nunique() < 3:
        continue
    a = roc_auc_score(yL, s.fillna(s.median()))
    rows.append([c, round(max(a, 1 - a), 3), "future/outcome" if ("last_30d" in c or c.startswith("trend_") or c in ("ctr",)) else ""])
scr = pd.DataFrame(rows, columns=["column", "single-column AUC (direction-free)", "note"]).sort_values("single-column AUC (direction-free)", ascending=False)
print("Top 12 single columns by ranking power:")
print(scr.head(12).to_string(index=False))

# ---- fold runner: held-out-client AUC for any feature list
def held_out_auc(feats, data=None, labels=None, extra=None):
    d0 = el if data is None else data
    preds, ys = [], []
    for f in range(5):
        trn = d0[(d0["fold"] != f) & d0["labelled"]]; tst = d0[(d0["fold"] == f) & d0["labelled"]]
        typ = band_typical(trn)
        trn = add_relative(trn, typ); tst = add_relative(tst, typ)
        ytr = trn["y"].values if labels is None else labels.loc[trn.index].values
        yte = tst["y"].values if labels is None else labels.loc[tst.index].values
        m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED).fit(trn[feats].fillna(0), ytr)
        preds.append(m.predict_proba(tst[feats].fillna(0))[:, 1]); ys.append(yte)
    return round(roc_auc_score(np.concatenate(ys), np.concatenate(preds)), 3)

results = {}
results["reference (my 17 features)"] = held_out_auc(FEATURES)

# ---- Attack 2: ablation
results["drop avg_position"] = held_out_auc([f for f in FEATURES if f != "avg_position"])

# ---- Attack 3: add the forbidden columns
el["days_since_last_update"] = el["days_since_last_update"].fillna(0)
results["+ days_since_last_update"] = held_out_auc(FEATURES + ["days_since_last_update"])
el["trend_pct"] = el["trend_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
results["+ trend_pct (outcome-looking)"] = held_out_auc(FEATURES + ["trend_pct"])

# ---- Attack 4: controls
el["LEAK_ctr_last"] = el["ctr_last"].replace([np.inf, -np.inf], np.nan).fillna(0)
results["deliberate leak (+ later CTR)"] = held_out_auc(FEATURES + ["LEAK_ctr_last"])
rng = np.random.default_rng(SEED)
shuf = pd.Series(el["y"].values.copy(), index=el.index)
idx = el.index[el["labelled"]]
shuf.loc[idx] = rng.permutation(shuf.loc[idx].values)
results["shuffled labels"] = held_out_auc(FEATURES, labels=shuf)

res = pd.DataFrame({"held-out-client AUC": results})
display(res)

Top 12 single columns by ranking power:
              column  single-column AUC (direction-free)           note
     clicks_last_30d                               0.874 future/outcome
                 ctr                               0.874 future/outcome
          clicks_90d                               0.796               
     clicks_prev_30d                               0.735               
  days_with_sessions                               0.723               
       pageviews_90d                               0.693               
        sessions_90d                               0.686               
           users_90d                               0.683               
   sessions_last_30d                               0.679 future/outcome
   sessions_prev_30d                               0.670               
   scroll_events_90d                               0.663               
engaged_sessions_90d                               0.658               


,held-out-client AUC
reference (my 17 features),0.789
drop avg_position,0.784
+ days_since_last_update,0.784
+ trend_pct (outcome-looking),0.798
deliberate leak (+ later CTR),1.000
shuffled labels,0.495


**Reading the hunt (and what I would do about each result):**

- **Reference score is the number to beat**; it is the same model as Week 5, about 0.79 AUC on clients it never saw.
- **Dropping `avg_position` barely moves it.** The model is not secretly living on one column.
- **Adding `days_since_last_update` changes nothing (0.784) and `trend_pct` nudges the score up only slightly (0.798)**, so neither is an obvious shortcut *in this snapshot*. I still **exclude** them (next section): "not obviously leaking in one test" is not the same as "safe to use", and `trend_pct` is computed from the later window by construction.
- **The deliberate leak lands near 1.0** (the test can see a leak when there is one) **and shuffled labels land near 0.5** (the pipeline cannot invent signal). Both controls behave, which is what makes the other numbers believable.
- **Column screen is where the real red flags show up.** `clicks_last_30d` and `ctr` (both built from the later window) rank at about 0.87 on their own, and **`clicks_90d` alone reaches about 0.80, higher than my whole 17-feature model (0.79)**. A 90-day total contains the later 30 days, so that is leakage, not skill. My allowed columns (e.g. `clicks_prev_30d`, about 0.74) stay below the model. That one table is why the 90-day totals are on the exclusion list.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
excluded = pd.DataFrame([
    ["impressions_last_30d, clicks_last_30d, sessions_last_30d", "the LATER window: this is the future (and the label is built from it)"],
    ["ctr, trend_direction, trend_pct", "computed from or summarising the movement I am predicting (outcome / label risk)"],
    ["impressions_90d, clicks_90d, sessions_90d, pageviews_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d", "90-day totals overlap the later window, so they contain the future"],
    ["days_with_impressions, days_with_sessions", "counted across the full 90 days, so they overlap the later window"],
    ["days_since_last_update, freshness_tier", "when a page was edited can follow the traffic change (people edit pages that already fell); the error analysis showed the wrong picks were recently edited"],
    ["age_tier, impression_tier, position_tier, word_count_tier, char_count_tier, competition_level", "binned copies of fields I already use (no new information; some bin on 90-day values)"],
    ["engagement_rate, scroll_rate, ai_traffic_pct", "built from 90-day totals (future overlap); missing-heavy"],
    ["provider_used, model_used", "how the page was generated: a process detail, not a page-performance cause I can defend"],
    ["content_id, client_id", "identifiers: used only to group folds and sort ties, never as features (a model could memorise a client)"],
    ["any FlyRank product score or flag (health score, quick-win tag, priority score)", "not in the data; if rebuilt they are decisions, not inputs (they would hide the rule I am trying to beat)"],
], columns=["fields refused", "one-line why"])
display(excluded.style.hide(axis="index"))

# ---- privacy check: nothing in the data can be turned into a name, URL or query
obj = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
print("Text columns in the file:", obj)
for c in ["content_id", "client_id"]:
    s = df[c].astype(str)
    print(f"{c}: {s.nunique():,} distinct values, length {s.str.len().min()}-{s.str.len().max()}, any 'http' or '.com': {bool(s.str.contains('http|\\.com', regex=True).any())}")
print("No URL, query text, client name or domain column exists in this file; IDs are hashed and are published only inside aggregates.")

fields refused,one-line why
"impressions_last_30d, clicks_last_30d, sessions_last_30d",the LATER window: this is the future (and the label is built from it)
"ctr, trend_direction, trend_pct",computed from or summarising the movement I am predicting (outcome / label risk)
"impressions_90d, clicks_90d, sessions_90d, pageviews_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d","90-day totals overlap the later window, so they contain the future"
"days_with_impressions, days_with_sessions","counted across the full 90 days, so they overlap the later window"
"days_since_last_update, freshness_tier",when a page was edited can follow the traffic change (people edit pages that already fell); the error analysis showed the wrong picks were recently edited
"age_tier, impression_tier, position_tier, word_count_tier, char_count_tier, competition_level",binned copies of fields I already use (no new information; some bin on 90-day values)
"engagement_rate, scroll_rate, ai_traffic_pct",built from 90-day totals (future overlap); missing-heavy
"provider_used, model_used","how the page was generated: a process detail, not a page-performance cause I can defend"
"content_id, client_id","identifiers: used only to group folds and sort ties, never as features (a model could memorise a client)"
"any FlyRank product score or flag (health score, quick-win tag, priority score)","not in the data; if rebuilt they are decisions, not inputs (they would hide the rule I am trying to beat)"


Text columns in the file: ['content_id', 'client_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']
content_id: 30,000 distinct values, length 20-20, any 'http' or '.com': False
client_id: 32 distinct values, length 17-17, any 'http' or '.com': False
No URL, query text, client name or domain column exists in this file; IDs are hashed and are published only inside aggregates.


**Rule I follow.** A field is allowed only if I can say, in one line, *when it exists and how it was produced*. If I cannot, it goes on this list. The list is deliberately long: leaving out a useful column costs a little accuracy; leaving in a leaking one makes every number in the paper wrong.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.